# 🎬 AI Scene Generator - AnimateDiff

Generate full animated scenes with character movement, emotions, and lip-sync.

**Instructions:**
1. Click **Runtime → Run all**
2. Upload your character image
3. Upload your audio file
4. Enter a description prompt
5. Wait for generation (10-30 minutes)
6. Download the result

**Free GPU:** Uses Google Colab's free GPU. Be patient - quality takes time!

In [ ]:
# Install only what's needed (use Colab's pre-installed torch)
!pip install -q diffusers transformers accelerate xformers einops omegaconf safetensors
!pip install -q imageio imageio-ffmpeg moviepy
print("✓ Dependencies installed")

In [ ]:
# Upload files first
from google.colab import files

print("📸 Upload your CHARACTER IMAGE:")
uploaded = files.upload()
character_image = list(uploaded.keys())[0]
print(f"✓ Character image: {character_image}")

print("\n🎵 Upload your AUDIO FILE:")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"✓ Audio file: {audio_file}")

In [ ]:
# Get user prompt
prompt = input("\n✍️ Describe your scene: ")
if not prompt:
    prompt = "animated skeleton character speaking with emotions, dramatic lighting, full body, cinematic"
print(f"Prompt: {prompt}")

In [ ]:
# Generate video with AnimateDiff
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
from diffusers.utils import export_to_video
from PIL import Image

print("🎬 Loading AnimateDiff...")

# Use working model combination
adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-2",
    torch_dtype=torch.float16
)

model_id = "SG161222/Realistic_Vision_V5.1_noVAE"

pipe = AnimateDiffPipeline.from_pretrained(
    model_id,
    motion_adapter=adapter,
    torch_dtype=torch.float16
)

scheduler = DDIMScheduler.from_pretrained(
    model_id,
    subfolder="scheduler",
    clip_sample=False,
    timestep_spacing="linspace",
    beta_schedule="linear",
    steps_offset=1,
)
pipe.scheduler = scheduler

# Optimize for free GPU
pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

print("✓ Pipeline ready")
print("\n🎨 Generating animation (this takes 10-30 minutes)...")

# Generate
output = pipe(
    prompt=prompt,
    negative_prompt="blurry, bad quality, distorted, ugly, deformed",
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=20,
    generator=torch.Generator("cpu").manual_seed(42)
)

frames = output.frames[0]
export_to_video(frames, "animated_scene.mp4", fps=8)

print("✓ Animation generated!")

In [ ]:
# Add audio to video
from moviepy.editor import VideoFileClip, AudioFileClip

print("🎵 Adding audio...")

video = VideoFileClip("animated_scene.mp4")
audio = AudioFileClip(audio_file)

# Loop video to match audio if needed
if audio.duration > video.duration:
    loops = int(audio.duration / video.duration) + 1
    from moviepy.editor import concatenate_videoclips
    video = concatenate_videoclips([video] * loops).subclip(0, audio.duration)

final = video.set_audio(audio)
final.write_videofile("final_scene.mp4", codec="libx264", audio_codec="aac", logger=None)

video.close()
audio.close()
final.close()

print("✓ Final video ready!")

In [ ]:
# Preview and download
from IPython.display import Video
from google.colab import files

print("\n📺 Preview:")
display(Video("final_scene.mp4", width=512))

print("\n📥 Downloading...")
files.download("final_scene.mp4")

print("\n✅ DONE! Your AI-generated animated scene is ready!")
print("\nTo generate another scene, run all cells again.")